# Exercise 4 – Maximum entropy under mean constraints

Invent a real-world example representing four possible computational costs (you MUST present an example not only the 4 numerical values). (Order for increasing values of $x_i$).

A corporation has a pipeline for elaborating system logs that can be implemented in four languages, $x$ is the time of the CPU per job:
- $x_1 = 10$ ms, C++
- $x_2 = 15$ ms, Rust
- $x_3 = 25$ ms, Go
- $x_4 = 100$ ms, Python


## Question 1
Compute numerically the MaxEnt pmf for three different values of $µ$.

## Answer 1


Given $\Omega_X\{10, 15, 25, 100\}$ with average $\bar x = 37.5$ ms.


We have selected these three values of $\mu$:
- $\mu_1 = 12$
- $\mu_2 = 37.5$
- $\mu_3 = 75$


According to the Principle of Maximum Entropy (MaxEnt) under a known mean constraint $\mu = \mathbb{E}[X]$, the probability mass function (PMF) belongs to the exponential family:
$$p_i = \frac{\beta^{x_i}}{\sum_{j} \beta^{x_j}}$$
where $\beta > 0$ is determined uniquely by solving the mean constraint equation:
$$\sum_{i} (x_i - \mu) \beta^{x_i} = 0$$

Computation:



````
% Esercizio 4 - Soluzione MATLAB
x = [10, 15, 25, 100];
% Scelgo 3 medie: una bassa (12), la media esatta per la distribuzione uniforme (37.5) e una alta (75)
mus = [12, 37.5, 75]; 
names = {'C++','Rust','Go','Python'};
% Impostiamo max_range a 2, poichè per mu=75, la soluzione di beta sarà > 1.
max_range = 2; 

syms beta
figure;
for k = 1:3
    mu = mus(k);
    
    % Equazione di Lagrange derivata dalla forma p_i = (beta^x_i) / Z
    % Questo assicura che il vincolo beta > 0 sia sempre matematicamente coerente
    eqn = ( sum(x.*(beta.^x)) / sum(beta.^x) - mu ) == 0;
    
    % Risoluzione numerica con vpasolve rispettando i limiti [0, max_range]
    beta_sol = vpasolve(eqn, beta, [0 max_range]);

    if isempty(beta_sol)
        error('Nessuna soluzione ammissibile (beta > 0) per mu = %g', mu);
    end
    
    % Estrarre il valore reale positivo
    beta_val = double(beta_sol(1));

    % Calcolo della Probability Mass Function (pmf)
    p = (beta_val.^x); 
    p = double(p / sum(p));
    
    % Calcolo dell'Entropia (H) in bit
    H = -sum(p.*log2(p + eps));
    
    fprintf('mu=%g  beta=%.5f  H=%.4f\n', mu, beta_val, H);

    subplot(1,3,k); 
    bar(categorical(names,names), p);
    title(sprintf('\\mu=%g, \\beta=%.4f, H=%.3f', mu, beta_val, H));
    ylabel('p_i'); 
    ylim([0 1]);
end
````

## Question 3: Which value of $\mu$ produces the uniform distribution?
The uniform distribution is produced when $\mu = 37.5$. 
A uniform distribution occurs when all states are equally likely (i.e., $p_i = 1/4$ for all $i$). Looking at the formula $p_i \propto \beta^{x_i}$, the only way the probabilities can be equal regardless of their $x_i$ value is if $\beta = 1$. 
If we set $\beta = 1$ in the constraint equation $\sum (x_i - \mu)\beta^{x_i} = 0$, it simplifies to $\sum (x_i - \mu) = 0$, meaning $\mu = \frac{1}{N} \sum x_i$. In our example, the arithmetic average is $(10 + 15 + 25 + 100) / 4 = 150 / 4 = 37.5$. Therefore, constraining the mean to exactly the simple average yields the uniform distribution.

## Question 4: Comparison and interpretation of the mean constraint
- **When $\mu = 12$**: The mean constraint is much lower than the simple average. The optimization forces $\beta < 1$, which exponentially heavily penalizes higher costs. This creates a severely right-skewed distribution where the lowest cost (C++) dominates to pull the average down, resulting in lower entropy.
- **When $\mu = 37.5$**: The constraint perfectly matches the uniform average. The distribution is flat ($\beta=1$), perfectly balanced, and reaches the maximum possible entropy since the constraint does not force the distribution in any particular direction.
- **When $\mu = 75$**: The mean constraint is very high. The optimization yields $\beta > 1$, meaning higher exponents $x_i$ lead to exponentially larger weights. This creates a left-skewed distribution where Python (100ms) dominates in order to satisfy the large average, again reducing the overall entropy.
**Interpretation**: The mean constraint acts as an anchor. As the constraint $\mu$ moves away from the uniform average (37.5) towards the extremes of the domain, it forces the probability mass to concentrate around those extremes, thereby changing the shape of the PMF and lowering the system's entropy (uncertainty).

## Question 5: Does MaxEnt necessarily recover the true distribution?
No, MaxEnt does **

No, the Maximum Entropy principle guarantees only the derivation of the most neutral, unbiased, and least informative distribution possible, based solely on the explicitly known constraints included in the system (in this exercise, the only information provided to the model was the arithmetic mean $\mu$ of the computational costs). However, in reality, the process generating logs and distributing work among C++, Rust, Go, and Python within a corporate pipeline is likely subject to numerous other practical constraints that are not known *a priori* and are not part of the formulated Lagrange equation. Such hidden constraints could include strict latency limits, team preferences for a specific corporate language, licensing restrictions, or bottlenecks regarding hardware core availability. If these constraints exist, the "true" empirical data distribution will exhibit systematically lower entropy and a different shape compared to the theoretical MaxEnt model. The MaxEnt approach is the optimal inferential tool for operating under uncertainty without introducing false biases, but it offers no guarantee of "guessing" or recovering the underlying reality when critical information is missing from the model.